In [ ]:
import numpy as np
import pandas as pd

class QuantumFleetOptimizer:
    def __init__(self, num_routes, num_ships, pop_size=50):
        self.num_routes = num_routes
        self.num_ships = num_ships
        self.pop_size = pop_size
        
        # Initialize Q-bits (probability amplitudes alpha and beta)
        # State vector initialized to 1/sqrt(2) for equal superposition |+>
        self.q_population = np.ones((pop_size, num_routes, num_ships, 2)) / np.sqrt(2)

    def measure_superposition(self):
        """Collapse Q-bits into classical binary routing decisions (0 or 1)."""
        # The probability of state |1> is the square of the beta amplitude
        probabilities = self.q_population[:, :, :, 1] ** 2
        random_draws = np.random.rand(self.pop_size, self.num_routes, self.num_ships)
        return (random_draws < probabilities).astype(int)

    def evaluate_fitness(self, classical_states, df, weights=(0.4, 0.4, 0.2)):
        """Score each deployment using your dataset columns."""
        scores = np.zeros(self.pop_size)
        w_energy, w_emissions, w_cost = weights
        
        for i, deployment in enumerate(classical_states):
            total_cost = 0
            # For each active assignment, extract costs from your dataframe
            for route_idx in range(self.num_routes):
                for ship_idx in range(self.num_ships):
                    if deployment[route_idx, ship_idx] == 1:
                        # In production, locate the exact row matching this ship/route combo
                        # row = df[(df['Route_Type'] == route_idx) & (df['Ship_Type'] == ship_idx)]
                        
                        # Placeholder lookups based on your columns:
                        simulated_energy = 15000  # df['Energy_Consumed_kWh']
                        simulated_emissions = 500 # df['Lifecycle_Emissions_kgCO2e']
                        simulated_cost = 2000     # df['Operational_Cost_USD']
                        
                        total_cost += (w_energy * simulated_energy + 
                                       w_emissions * simulated_emissions + 
                                       w_cost * simulated_cost)
            
            scores[i] = total_cost
        return scores

    def quantum_rotation_gate(self, best_deployment, theta=0.05 * np.pi):
        """Evolve the population amplitudes toward the lowest-cost routing."""
        for i in range(self.pop_size):
            for r in range(self.num_routes):
                for s in range(self.num_ships):
                    alpha, beta = self.q_population[i, r, s]
                    target_bit = best_deployment[r, s]

                    # Determine the rotation direction based on the best known state
                    direction = 1 if (alpha * beta > 0 and target_bit == 1) else -1
                    if target_bit == 0 and alpha * beta < 0: direction = 1
                    elif target_bit == 0 and alpha * beta > 0: direction = -1
                    
                    d_theta = direction * theta

                    # Apply the standard 2D rotation matrix
                    new_alpha = alpha * np.cos(d_theta) - beta * np.sin(d_theta)
                    new_beta = alpha * np.sin(d_theta) + beta * np.cos(d_theta)

                    self.q_population[i, r, s] = [new_alpha, new_beta]

In [ ]:
# Extract unique labels from your dataset
unique_routes = df['Route_Type'].unique()
unique_ships = df['Ship_Type'].unique()

# Create bi-directional mappings for routes
route_to_idx = {route: i for i, route in enumerate(unique_routes)}
idx_to_route = {i: route for i, route in enumerate(unique_routes)}

# Create bi-directional mappings for ships
ship_to_idx = {ship: i for i, ship in enumerate(unique_ships)}
idx_to_ship = {i: ship for i, ship in enumerate(unique_ships)}

print(f"Tracking {len(unique_routes)} unique routes and {len(unique_ships)} unique ships.")

In [ ]:
def evaluate_fitness(self, classical_states, df, idx_to_route, idx_to_ship, weights=(0.4, 0.4, 0.2)):
        """Score each deployment by mapping indices back to dataframe categories."""
        scores = np.zeros(self.pop_size)
        w_energy, w_emissions, w_cost = weights
        
        for i, deployment in enumerate(classical_states):
            total_cost = 0
            
            for route_idx in range(self.num_routes):
                for ship_idx in range(self.num_ships):
                    # If this ship is assigned to this route
                    if deployment[route_idx, ship_idx] == 1:
                        
                        # 1. Translate matrix indices back to your dataset strings
                        target_route = idx_to_route[route_idx]
                        target_ship = idx_to_ship[ship_idx]
                        
                        # 2. Look up the exact physical metrics for this combination
                        # Assuming the dataframe has a row for this specific ship/route combo
                        match = df[(df['Route_Type'] == target_route) & (df['Ship_Type'] == target_ship)]
                        
                        if not match.empty:
                            # Extract your exact optimization targets
                            energy = match['Energy_Consumed_kWh'].values[0]
                            emissions = match['Lifecycle_Emissions_kgCO2e'].values[0]
                            cost = match['Operational_Cost_USD'].values[0]
                            
                            total_cost += (w_energy * energy + w_emissions * emissions + w_cost * cost)
                        else:
                            # Heavy penalty if this ship cannot physically sail this route
                            total_cost += 9999999 
            
            scores[i] = total_cost
            
        return scores

In [ ]:
optimizer = QuantumFleetOptimizer(
    num_routes=len(unique_routes), 
    num_ships=len(unique_ships), 
    pop_size=50
)

In [ ]:
# Initialize a mask of all 1s (all assignments initially assumed valid)
feasibility_mask = np.ones((len(unique_routes), len(unique_ships)), dtype=int)

# Sweep the dataframe to apply physical constraints
for r, route_name in idx_to_route.items():
    for s, ship_name in idx_to_ship.items():
        match = df[(df['Route_Type'] == route_name) & (df['Ship_Type'] == ship_name)]
        
        if not match.empty:
            # Extract weights
            max_cap = match['Max_Capacity'].values[0]
            cargo_weight = match['Cargo_Weight_tons'].values[0]
            
            # If the ship physically cannot carry the cargo, lock the mask to 0
            if cargo_weight > max_cap:
                feasibility_mask[r, s] = 0
        else:
            # If the combo doesn't exist in the data, lock it out
            feasibility_mask[r, s] = 0

In [ ]:
def __init__(self, num_routes, num_ships, feasibility_mask, pop_size=50):
        self.num_routes = num_routes
        self.num_ships = num_ships
        self.pop_size = pop_size
        self.feasibility_mask = feasibility_mask # Inject the mask here
        
        # Initialize Q-bits
        self.q_population = np.ones((pop_size, num_routes, num_ships, 2)) / np.sqrt(2)

def measure_superposition(self):
		"""Collapse Q-bits and strictly enforce physical constraints."""
		probabilities = self.q_population[:, :, :, 1] ** 2
		random_draws = np.random.rand(self.pop_size, self.num_routes, self.num_ships)
		
		# 1. Standard quantum measurement
		raw_states = (random_draws < probabilities).astype(int)
		
		# 2. Apply the strict constraint mask (Broadcasting across the population)
		# Any raw_state = 1 that hits a mask = 0 instantly becomes 0
		constrained_states = raw_states * self.feasibility_mask
		
		return constrained_states

In [ ]:
# ==========================================
# PHASE 5: QUANTUM-INSPIRED EVOLUTION
# ==========================================
import numpy as np

# 1. Define Optimization Hyperparameters
N_GENERATIONS = 100
POPULATION_SIZE = 50

# 2. Instantiate the Optimizer
optimizer = QuantumFleetOptimizer(
    num_routes=len(unique_routes), 
    num_ships=len(unique_ships), 
    feasibility_mask=feasibility_mask,
    pop_size=POPULATION_SIZE
)

# 3. Tracking Variables
global_best_cost = float('inf')
global_best_deployment = None
convergence_history = []

print("Starting Quantum-Inspired Optimization Engine...\n")

# 4. The Main Loop
for generation in range(N_GENERATIONS):
    
    # Step A: Collapse Q-bits into classical, mask-repaired routing states
    classical_states = optimizer.measure_superposition()
    
    # Step B: Evaluate the fitness using your dataset lookups
    scores = optimizer.evaluate_fitness(
        classical_states, 
        df, 
        idx_to_route, 
        idx_to_ship, 
        weights=(0.4, 0.4, 0.2) # Energy, Emissions, Cost weights
    )
    
    # Step C: Identify the elite chromosome of this generation
    min_idx = np.argmin(scores)
    current_best_cost = scores[min_idx]
    current_best_deployment = classical_states[min_idx]
    
    # Step D: Update the global historical best
    if current_best_cost < global_best_cost:
        global_best_cost = current_best_cost
        global_best_deployment = np.copy(current_best_deployment)
        
    convergence_history.append(global_best_cost)
    
    # Step E: Apply the Quantum Rotation Gate
    # Dynamic theta creates a "simulated annealing" effect: 
    # Large steps early on for exploration, small steps later for precise convergence.
    dynamic_theta = 0.05 * np.pi * (1 - (generation / N_GENERATIONS)) 
    optimizer.quantum_rotation_gate(global_best_deployment, theta=dynamic_theta)
    
    # Step F: Log Progress
    if generation % 10 == 0 or generation == N_GENERATIONS - 1:
        print(f"Generation {generation:3d} | Optimal Multi-Objective Cost: {global_best_cost:,.2f}")

print("\nOptimization Complete.")

In [ ]:
import pandas as pd

def decode_deployment_to_dataframe(best_deployment, df, idx_to_route, idx_to_ship):
    """Translates the binary optimization matrix back into a human-readable report."""
    optimized_routes = []
    
    num_routes, num_ships = best_deployment.shape
    
    for route_idx in range(num_routes):
        for ship_idx in range(num_ships):
            # If the optimizer decided this ship should take this route
            if best_deployment[route_idx, ship_idx] == 1:
                
                # 1. Map the integer indices back to your categorical strings
                target_route = idx_to_route[route_idx]
                target_ship = idx_to_ship[ship_idx]
                
                # 2. Extract the exact performance metrics from the original dataset
                match = df[(df['Route_Type'] == target_route) & (df['Ship_Type'] == target_ship)]
                
                if not match.empty:
                    energy = match['Energy_Consumed_kWh'].values[0]
                    emissions = match['Lifecycle_Emissions_kgCO2e'].values[0]
                    cost = match['Operational_Cost_USD'].values[0]
                    
                    # 3. Append to our report list
                    optimized_routes.append({
                        'Ship_Type': target_ship,
                        'Assigned_Route': target_route,
                        'Operational_Cost_USD': cost,
                        'Lifecycle_Emissions_kgCO2e': emissions,
                        'Energy_Consumed_kWh': energy
                    })
    
    # Convert the raw list into a formatted Pandas DataFrame
    report_df = pd.DataFrame(optimized_routes)
    
    # Sort by cost so the most expensive deployments are at the top
    report_df = report_df.sort_values(by='Operational_Cost_USD', ascending=False).reset_index(drop=True)
    
    return report_df

In [ ]:
# Generate the final report
final_deployment_report = decode_deployment_to_dataframe(
    global_best_deployment, 
    df, 
    idx_to_route, 
    idx_to_ship
)

# Display the optimized fleet routing
print("\n=== FINAL OPTIMIZED FLEET DEPLOYMENT ===")
print(final_deployment_report.to_string())

# Optional: Save it straight to a CSV for your deliverable submission
final_deployment_report.to_csv("Optimized_Green_Fleet_Deployment.csv", index=False)